# 02 - Predicting listing price with linear regression

This notebook does three things:

1. reproduces the **original** approach from the course submission (label-encoded categories, IQR filter, plain `LinearRegression` on raw Toman) and shows why its number cannot be trusted;
2. fits the **revised** model: split first, one-hot encode brand / condition / origin / colour, standardise the numeric columns, predict `log(price)`, compare `LinearRegression` with `Ridge`;
3. reports R², MAE and RMSE in Toman on a held-out test split next to a median-price baseline.

`src/train.py` runs the same revised pipeline from the command line and saves the model the app uses.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

from src.features import CATEGORICAL, FEATURES, NUMERIC, TARGET, RAW_PATH, CLEAN_PATH, COLUMN_MAP, NOT_SPECIFIED
from src.features import build_preprocessor, parse_capacity_gb, parse_price_toman, parse_sim_count

df = pd.read_csv(CLEAN_PATH)
print(df.shape)
df.head()

## A quick look at the target

Prices span more than two orders of magnitude and are right-skewed. On a log scale the distribution is close to symmetric, and brand explains a large part of the spread. That is why the revised model predicts `log(price)`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df[TARGET] / 1e6, bins=50)
axes[0].set_xlabel("price (million Toman)"); axes[0].set_title("raw price")
axes[1].hist(np.log10(df[TARGET]), bins=50)
axes[1].set_xlabel("log10(price)"); axes[1].set_title("log price")
plt.tight_layout(); plt.show()

In [ ]:
top_brands = df["Brand"].value_counts().index[:6]
plt.figure(figsize=(9, 4))
sns.boxplot(data=df[df["Brand"].isin(top_brands)], x="Brand", y=TARGET, order=top_brands)
plt.yscale("log"); plt.ylabel("price (Toman, log scale)"); plt.title("price by brand, six most common brands")
plt.show()

## Original approach (for the record)

The course submission label-encoded `Brand and Model` (315 classes), `Color`, `Status` and `Brand Origin` into integers, filtered price by IQR on the whole table, split 90/10 and fitted `LinearRegression` on raw Toman. It reported a single number, MSE ≈ 1.87e14.

The cell below re-creates the original cleaning (no de-duplication, no whitespace normalisation, no price bounds) and the original model from the raw file, so the number is reproduced exactly. It then adds what was never reported: R², the number of negative predictions, and how many test rows also appear in the training set.

In [ ]:
def original_clean(raw):
    d = raw.rename(columns=COLUMN_MAP).dropna()
    d = d[~d.astype(str).apply(lambda c: c.str.contains(NOT_SPECIFIED)).any(axis=1)]
    d = d.assign(**{
        "SIM Count": d["SIM Count"].map(parse_sim_count),
        "Internal Storage(GB)": d["Internal Storage(GB)"].map(parse_capacity_gb),
        "RAM(GB)": d["RAM(GB)"].map(parse_capacity_gb),
        TARGET: d[TARGET].map(parse_price_toman),
        "Color": d["Color"].str.replace(r"[A-Za-z]+", "", regex=True),   # Latin words removed, whitespace kept
    })
    return d[d["Color"].str.strip() != ""]

orig = original_clean(pd.read_csv(RAW_PATH))
for col in ["Color", "Brand Origin", "Brand and Model", "Status"]:
    orig[col] = LabelEncoder().fit_transform(orig[col])

q1, q3 = orig[TARGET].quantile([0.25, 0.75])
iqr = q3 - q1
orig = orig[orig[TARGET].between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)]

X_o, y_o = orig.drop(columns=TARGET), orig[TARGET]
Xo_tr, Xo_te, yo_tr, yo_te = train_test_split(X_o, y_o, test_size=0.1, random_state=519)
orig_pred = LinearRegression().fit(Xo_tr, yo_tr).predict(Xo_te)

leaked = pd.merge(Xo_tr.assign(y=yo_tr), Xo_te.assign(y=yo_te), how="inner").shape[0]
print(f"rows after IQR filter: {len(orig)}   test rows: {len(Xo_te)}")
print(f"MSE  = {mean_squared_error(yo_te, orig_pred):.4e}   (reported in the original submission)")
print(f"RMSE = {np.sqrt(mean_squared_error(yo_te, orig_pred)) / 1e6:.2f}M Toman")
print(f"MAE  = {mean_absolute_error(yo_te, orig_pred) / 1e6:.2f}M Toman")
print(f"R2   = {r2_score(yo_te, orig_pred):.3f}")
print(f"negative predicted prices: {(orig_pred < 0).sum()} of {len(orig_pred)}")
print(f"test rows that are exact copies of a training row: {leaked} of {len(Xo_te)}")

The R² is not zero, but the number is not trustworthy: a quarter of the test rows are duplicates of training rows, the encoders and the IQR bounds were fitted on the test rows too, and the model happily predicts negative prices. Fitting a slope over an alphabetical ordering of 315 model names is also not a model of anything.

## Revised approach

* **Split first** (80/20, `random_state=42`). Everything below is fitted on the training split only.
* **Brand instead of model name.** `Brand` is the first token of "Brand and Model" (18 brands). The 315 model names on ~1,400 training rows would mostly be memorised.
* **One-hot encode** brand, condition, origin and colour; **standardise** SIM count, storage and RAM.
* **Predict `log(price)`** and transform back, so predictions are always positive and the loss is relative rather than dominated by the most expensive phones.
* Compare `LinearRegression` with `Ridge` (alpha chosen by 5-fold CV on the training split).

In [ ]:
X, y = df[FEATURES], df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"train rows: {len(X_train)}   test rows: {len(X_test)}")

def make_model(regressor):
    pipe = Pipeline([("prep", build_preprocessor()), ("reg", regressor)])
    return TransformedTargetRegressor(regressor=pipe, func=np.log, inverse_func=np.exp)

candidates = {
    "LinearRegression": make_model(LinearRegression()),
    "Ridge": make_model(RidgeCV(alphas=np.logspace(-3, 3, 25), cv=5)),
}

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_mae = {}
for name, model in candidates.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="neg_mean_absolute_error")
    cv_mae[name] = -scores.mean()
    print(f"{name:18s} 5-fold CV MAE on train = {cv_mae[name] / 1e6:.2f}M Toman")
winner_name = min(cv_mae, key=cv_mae.get)
print("selected on CV:", winner_name)

In [ ]:
def evaluate(y_true, y_pred):
    return {"R2": r2_score(y_true, y_pred),
            "MAE (M Toman)": mean_absolute_error(y_true, y_pred) / 1e6,
            "RMSE (M Toman)": np.sqrt(mean_squared_error(y_true, y_pred)) / 1e6}

results = {"median baseline": evaluate(y_test, np.full(len(y_test), y_train.median()))}
for name, model in candidates.items():
    model.fit(X_train, y_train)
    results[name] = evaluate(y_test, model.predict(X_test))

alpha = candidates["Ridge"].regressor_.named_steps["reg"].alpha_
print(f"Ridge alpha chosen by CV: {alpha:.4g}")
pd.DataFrame(results).T.round(3)

Ridge and plain least squares land on the same numbers: with 16 one-hot brand columns and three numeric ones there is nothing to regularise, and CV picks a tiny alpha. The plain model is kept.

In [ ]:
winner = candidates[winner_name]
y_pred = winner.predict(X_test)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test / 1e6, y_pred / 1e6, s=14, alpha=0.6)
ax.plot([0.4, 350], [0.4, 350], color="tab:red", label="perfect prediction")
ax.set(xscale="log", yscale="log", xlim=(0.4, 350), ylim=(0.4, 350),
       xlabel="actual price (million Toman)", ylabel="predicted price (million Toman)",
       title=f"{winner_name}: actual vs predicted (test split)")
ax.legend(); ax.grid(True, which="both", alpha=0.3)
plt.show()

## Which features matter?

With one-hot columns and a log target, each coefficient is a multiplicative effect: `exp(coef)` is the price ratio relative to the dropped reference category, holding the other features fixed.

In [ ]:
prep = winner.regressor_.named_steps["prep"]
reg = winner.regressor_.named_steps["reg"]
names = prep.get_feature_names_out()
coefs = pd.Series(reg.coef_, index=names).sort_values()
print("intercept (exp):", f"{np.exp(reg.intercept_):,.0f} Toman")
pd.DataFrame({"coef": coefs, "price ratio exp(coef)": np.exp(coefs)}).round(3)

## Where the model fails

Brand plus storage and RAM cannot separate an iPhone 7 from an iPhone 13 Pro Max: both are "اپل, 128 GB, 4-6 GB RAM". The horizontal bands in the scatter plot are exactly those groups. The model name carries most of the remaining price signal, and a dataset this small cannot learn 315 of them. See the README for the full list of limitations.